In [1]:
import requests
import xml.etree.ElementTree as ET
import polars as pl
import time
from AffFilter import extractUniqueAff
import pycountry


In [15]:
# Suchparameter
query = '"german"[Language] AND ("2025/01/01"[PDAT] : "2025/12/31"[PDAT])'
#alle Deutschen Veröffentlichungen des Jahres 2025
apiKey = "7e14fd2cb71ccbb3dade927e99df741a5009" #um auf 10 Veröffentlichungen zugreifen zu können
retmax = 50 #return maximum (anzahl der Ergebnisse, die die API maximal zurückgeben soll)

#aufruf von esearch
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db" : "pubmed",
    "term" : query,
    "retmax" : retmax,
    "retmode" : "json",
    "api_key" : apiKey
}
#auf PubMed nach dem Suchstring suchen und die ersten 50(retmax) einträge ausgeben

response = requests.get(url, params=params)
data = response.json()
pmids = data["esearchresult"]["idlist"]
print(f"{len(pmids)} PMIDs gefunden") #Ausgabe der gefundenen PMIDs
print(pmids)


50 PMIDs gefunden
['42208598', '41973606', '41971571', '41971570', '41971569', '41971564', '41971561', '41921923', '41688089', '41586736', '41586734', '41586733', '41586732', '41586730', '41586729', '41586726', '41586725', '41586722', '41586721', '41586720', '41586717', '41586715', '41586712', '41586710', '41586709', '41586704', '41586700', '41586698', '41586446', '41586084', '41586082', '41584498', '41584495', '41583135', '41583133', '41583132', '41571265', '41569278', '41569277', '41569276', '41569275', '41569274', '41569273', '41569272', '41569250', '41569249', '41569248', '41569247', '41569246', '41569245']


In [16]:
#efetch Aufruf Metadaten als XML laden
ids = " , ".join(pmids)
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
params = {
    "db" : "pubmed",
    "id" : ids,
    "retmode" : "xml",
    "api_key": apiKey
}
response = requests.get(url, params=params)
xmlData = response.text
print(xmlData[:500])  #Anzeigen der ersten 500 Zeichen


<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="In-Process" Owner="NLM"><PMID Version="1">42208598</PMID><DateRevised><Year>2026</Year><Month>05</Month><Day>29</Day></DateRevised><Article PubModel="Print-Electronic"><Journal><ISSN IssnType="Electronic">1438-9010</ISSN><JournalIssue CitedMedium="Internet"><Volume>198</Vo


In [17]:
#parsen der XML und bauen des Polars DataFrames
def parsePubmedXML(xmlText) :
    root = ET.fromstring(xmlText)
    articles = []
    for article in root.findall(".//PubmedArticle"):

            #PMID
        pmid = article.findtext(".//PMID", default="")

            #titel
        title = article.findtext(".//ArticleTitle", default="")
            #sollte kein Englischer Titel vorhanden sein, den Deutschen nehmen(woanders gespeichert)
        if not title or title == "[Not Available].":
            title = article.findtext(".//VernacularTitle", default="kein Titel verfügbar") #evtl schauen wie Titel noch gespeichert werden können

            #Autoren und Affiliationen
        authors = []
        affilliations = []

        for author in article.findall(".//Author"):
            last = author.findtext("LastName", default="")
            fore = author.findtext("ForeName", default="")
            #hier nochmal schauen
            if last:
                authors.append(f"{last} {fore}".strip())
            for affil in author.findall("AffiliationInfo/Affiliation"):
                if affil.text and affil.text not in affilliations:
                    affilliations.append(affil.text)

        #Erstautor
        firstAuthor = authors[0] if authors else ""

        #link
        link = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

        articles.append({
            "pmid" : pmid,
            "title" : title,
            "firstAuthor" : firstAuthor,
            "authors" : "; ".join(authors),
            "affiliations" : " | ".join(affilliations),
            "link": link
        })

    return articles

artikel = parsePubmedXML(xmlData)
print(f"{len(artikel)} Artikel geparsed")
print(artikel[0]) #ersten Artikel zur Kontrolle anzeigen

50 Artikel geparsed
{'pmid': '42208598', 'title': 'Correction: Occupational Prenatal Radiation Exposure and Occupational Safety: Position Paper for Pregnant and breastfeeding Personnel in Interventional Procedures with Ionizing Radiation.', 'firstAuthor': 'Becker Lena Sophie', 'authors': 'Becker Lena Sophie; Stein Thomas; Frisch Anne; Dewald Cornelia Lieselotte Angelika; Blum Sophia Freya Ulrike; Wintergerst Charlotte; Torsello Giovanni; Peperhove Matti Jonas; Pöhler Gesa; Staudacher Emona; Franke Mareike; Tritt Stephanie; Westphalen Kerstin; Bruners Philipp; Rohde Stefan; Gebauer Bernhard; Das Marco; Jungnickel Kerstin; Fiebich Martin; Katoh Marcus; Paprottka Philipp; Uder Michael; Wacker Frank K; Uller Wibke', 'affiliations': 'Institute for Diagnostic and Interventional Radiology, Hannover Medical School, Hannover, Germany. | Department of Diagnostic and Interventional Radiology, Medical Center - University of Freiburg, Faculty of Medicine, University Freiburg, Germany. | Clinic for 

In [18]:
#fehlerhebung zur überprüfung wie die Affiliationen ausgegeben werden

root = ET.fromstring(xmlData)
ersterArtikel = root.find(".//PubmedArticle")
for author in ersterArtikel.findall(".//Author"):
    print("Autor: ", author.findtext("LastName"))
    for affil in author.findall("AffiliationInfo/Affiliation"):
        print("  Affiliation:", affil.text)

Autor:  Becker
  Affiliation: Institute for Diagnostic and Interventional Radiology, Hannover Medical School, Hannover, Germany.
Autor:  Stein
  Affiliation: Department of Diagnostic and Interventional Radiology, Medical Center - University of Freiburg, Faculty of Medicine, University Freiburg, Germany.
Autor:  Frisch
  Affiliation: Clinic for Radiology, CVK, Charité University Hospital Berlin, Berlin, Germany.
Autor:  Dewald
  Affiliation: Institute for Diagnostic and Interventional Radiology, Hannover Medical School, Hannover, Germany.
Autor:  Blum
  Affiliation: Institute and Polyclinic for Diagnostic and Interventional Radiology, University Hospital Carl Gustav Carus Dresden, Dresden, Germany.
Autor:  Wintergerst
  Affiliation: Department of Diagnostic and Interventional Radiology, Medical Center - University of Freiburg, Faculty of Medicine, University Freiburg, Germany.
Autor:  Torsello
  Affiliation: Department of Clinical and Interventional Radiology, University Medical Center 

In [19]:
# DataFrame bauen
df = pl.DataFrame(artikel)

#\xa0 aus xml bereinigen (geschütztes leerzeichen)
df = df.with_columns([
    pl.col("affiliations").str.replace_all("\xa0", " "),
    pl.col("title").str.replace_all("\xa0", " ")
])

print(df.shape)
df.head()
#df.filter(pl.col("title") != "[Not Available].").head()

(50, 6)


pmid,title,firstAuthor,authors,affiliations,link
str,str,str,str,str,str
"""42208598""","""Correction: Occupational Prena…","""Becker Lena Sophie""","""Becker Lena Sophie; Stein Thom…","""Institute for Diagnostic and I…","""https://pubmed.ncbi.nlm.nih.go…"
"""41973606""","""Mykotoxine – Bestimmung von Af…","""Berger Marion""","""Berger Marion; Deharde Max; Ne…","""Bundesanstalt für Arbeitsschut…","""https://pubmed.ncbi.nlm.nih.go…"
"""41971571""","""Cadmium und seine anorganische…","""Hallier Ernst""","""Hallier Ernst; Drexler Hans; H…","""37136 Ebergötzen Deutschland. …","""https://pubmed.ncbi.nlm.nih.go…"
"""41971570""","""Tri-(2-ethylhexyl)trimellitat …","""Kuhlmann Laura""","""Kuhlmann Laura; Eckert Elisabe…","""Friedrich-Alexander-Universitä…","""https://pubmed.ncbi.nlm.nih.go…"
"""41971569""","""Glaswolle, Halbwertszeit < 40 …","""Hartwig Andrea""","""Hartwig Andrea""","""Institut für Angewandte Biowis…","""https://pubmed.ncbi.nlm.nih.go…"


In [20]:
df.write_csv("pubmed_rohdaten.csv")
print("Gespeichert!")

Gespeichert!


In [21]:
#abfrage Gesamtzahl ohne Daten zu laden
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db" : "pubmed",
    "term" : query,
    "retmode" : "json",
    "api_key" : apiKey
}
response = requests.get(url, params=params)
data = response.json()
total = int(data["esearchresult"]["count"])
print(f"Gesamtanzahl Artikel: {total}")

Gesamtanzahl Artikel: 6506


In [22]:
#Alle PMIDs werden als Batches geholt

allePmids = []
batchSize = 500

for start in range (0, total, batchSize):
    params = {
        "db" : "pubmed",
        "term":query,
        "retmax": batchSize,
        "retstart" : start,
        "retmode" : "json",
        "api_key" : apiKey
    }

    response = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params=params)
    data = response.json()
    pmidsBatch = data["esearchresult"]["idlist"]
    allePmids.extend(pmidsBatch)
    print(f"{len(allePmids)} / {total}")
    time.sleep(0.1)
allePmids = list(dict.fromkeys(allePmids))

print(f"Fertig - {len(allePmids)} PMIDs insgesamt")

500 / 6506
1000 / 6506
1500 / 6506
2000 / 6506
2500 / 6506
3000 / 6506
3500 / 6506
4000 / 6506
4500 / 6506
5000 / 6506
5500 / 6506
6000 / 6506
6500 / 6506
6506 / 6506
Fertig - 6506 PMIDs insgesamt


In [23]:
alleArtikel = []
fetchBatchSize = 100

for i in range(0, len(allePmids), fetchBatchSize):
    batch = allePmids[i:i + fetchBatchSize]
    ids = ",".join(batch)

    params = {
        "db" : "pubmed",
        "id" : ids,
        "retmode" : "xml",
        "api_key" : apiKey
    }
    response = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi", params=params)
    artikel = parsePubmedXML(response.text)
    alleArtikel.extend(artikel)
    print(f"{len(alleArtikel)} / {len(allePmids)}")
    time.sleep(0.1)
print(f"Fertig - {len(alleArtikel)} Artikel insgesamt")

100 / 6506
200 / 6506
300 / 6506
400 / 6506
500 / 6506
600 / 6506
700 / 6506
800 / 6506
900 / 6506
1000 / 6506
1100 / 6506
1200 / 6506
1300 / 6506
1400 / 6506
1500 / 6506
1600 / 6506
1700 / 6506
1800 / 6506
1900 / 6506
2000 / 6506
2100 / 6506
2200 / 6506
2300 / 6506
2400 / 6506
2500 / 6506
2600 / 6506
2700 / 6506
2800 / 6506
2900 / 6506
3000 / 6506
3100 / 6506
3200 / 6506
3300 / 6506
3400 / 6506
3500 / 6506
3600 / 6506
3700 / 6506
3800 / 6506
3900 / 6506
4000 / 6506
4100 / 6506
4200 / 6506
4300 / 6506
4400 / 6506
4500 / 6506
4600 / 6506
4700 / 6506
4800 / 6506
4900 / 6506
5000 / 6506
5100 / 6506
5200 / 6506
5300 / 6506
5400 / 6506
5500 / 6506
5600 / 6506
5700 / 6506
5800 / 6506
5900 / 6506
6000 / 6506
6100 / 6506
6200 / 6506
6300 / 6506
6400 / 6506
6500 / 6506
6506 / 6506
Fertig - 6506 Artikel insgesamt


In [24]:
test = [a for a in alleArtikel if a["title"] == "kein Titel verfügbar"]
print(f"Artikel ohne Title: {len(test)}")

test2= [a for a in alleArtikel if a["title"] == "[Not Available]."]
print(f"Artikel mit [Not Available]: {len(test2)}")

Artikel ohne Title: 0
Artikel mit [Not Available]: 0


In [25]:
df = pl.DataFrame(alleArtikel)

df = df.with_columns([
    pl.col("affiliations").str.replace_all("\xa0", " "),
    pl.col("title").str.replace_all("\xa0", " ")
])

df.write_csv("pubmed_rohdaten_komplett.csv")

print(f"Gespeichert – {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
df.head()
print(df.shape)

Gespeichert – 6506 Zeilen, 6 Spalten
(6506, 6)


In [26]:
affiliations = extractUniqueAff(df)
print(f"Einzigartige Aff:{len(affiliations)}")
print((affiliations == "").sum())

Einzigartige Aff:14032
0


Country(alpha_2='DE', alpha_3='DEU', flag='🇩🇪', name='Germany', numeric='276', official_name='Federal Republic of Germany')
None
